# TP03 — Arboles de decisión - Ejercicio 2
   Bases de Datos Masivas (11088) · Gustavo An Contardi - 182818
   

## a. Tema del dataset

El dataset trata sobre cómo determinar el grado de un glioma, que es el tumor primario
más común del cerebro. Se clasifican en dos grados: LGG (glioma de bajo grado) y GBM
(glioblastoma multiforme), el de alto grado. Normalmente el grado se define con
criterios histológicos y de imágenes, pero los factores moleculares también son
importantes, y los estudios moleculares son caros.

Los datos vienen de los proyectos TCGA-LGG y TCGA-GBM de The Cancer Genome Atlas. Son
839 pacientes con tres datos clínicos (sexo, edad al diagnóstico y raza) y el estado
(mutado o no) de los 20 genes que mutan con más frecuencia en estos tumores. La tarea es
predecir si el paciente tiene LGG o GBM. Según la ficha de UCI, el objetivo es encontrar
el subconjunto de genes y datos clínicos que alcanza para clasificar bien, para mejorar
el resultado y bajar costos. No tiene valores faltantes.

Fuente: Tasci, E., Camphausen, K., Krauze, A. y Zhuge, Y. (2022). *Glioma Grading
Clinical and Mutation Features* [Dataset]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5R62J

## b. Árbol de decisión con hold-out 80/20

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

# Una sola semilla para todo: split, folds y árbol.
# Así, si dos corridas difieren, es por los parámetros y no por el azar.
SEMILLA = 42

df = pd.read_csv("data/gliomas.csv")
print(df.shape)
print(df["grado"].value_counts())
df["raza"].value_counts()

(839, 24)
grado
LGG    487
GBM    352
Name: count, dtype: int64


raza
Blanca                   765
Negra o afroamericana     59
Asiática                  14
Indígena americana         1
Name: count, dtype: int64

In [2]:
# Clase positiva = GBM. Precision, recall y F1 se calculan sobre la clase 1,
# así que esto define qué estamos midiendo.
y = (df["grado"] == "GBM").astype(int)

X = df.drop(columns="grado")
# sexo es binaria: con una columna 0/1 alcanza
X["sexo"] = (X["sexo"] == "Masculino").astype(int)
# raza va en one-hot: una columna por categoría, sin inventar orden.
# No uso drop_first: en un árbol la colinealidad no molesta y así se lee mejor.
X = pd.get_dummies(X, columns=["raza"], prefix="raza", dtype=int)

print(X.shape)
X.columns.tolist()

(839, 26)


['sexo',
 'edad_diagnostico',
 'IDH1',
 'TP53',
 'ATRX',
 'PTEN',
 'EGFR',
 'CIC',
 'MUC16',
 'PIK3CA',
 'NF1',
 'PIK3R1',
 'FUBP1',
 'RB1',
 'NOTCH1',
 'BCOR',
 'CSMD3',
 'SMARCA4',
 'GRIN2A',
 'IDH2',
 'FAT4',
 'PDGFRA',
 'raza_Asiática',
 'raza_Blanca',
 'raza_Indígena americana',
 'raza_Negra o afroamericana']

In [3]:
# stratify=y mantiene la proporción LGG/GBM igual en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

print("train:", X_train.shape, "  test:", X_test.shape)
# Control de que la estratificación funcionó
print(pd.DataFrame({"train": y_train.value_counts(normalize=True),
                    "test": y_test.value_counts(normalize=True)}).round(3))
# El caso único de "Indígena americana": ¿en qué lado cayó?
print("Indígena en train:", X_train["raza_Indígena americana"].sum(),
      " en test:", X_test["raza_Indígena americana"].sum())

train: (671, 26)   test: (168, 26)
       train   test
grado              
0       0.58  0.583
1       0.42  0.417
Indígena en train: 1  en test: 0


In [4]:
def evaluar(modelo, X, y):
    """Las cinco métricas que vamos a mirar, en un dict."""
    pred = modelo.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "precision_GBM": precision_score(y, pred),
        "recall_GBM": recall_score(y, pred),
        "f1_GBM": f1_score(y, pred),
    }

def matriz(modelo, X, y):
    """Matriz de confusión con etiquetas legibles en vez de 0/1."""
    cm = confusion_matrix(y, modelo.predict(X), labels=[0, 1])
    return pd.DataFrame(cm, index=["real LGG", "real GBM"], columns=["pred LGG", "pred GBM"])

# Árbol con parámetros por defecto: crece hasta que todas las hojas son puras.
# Es la línea de base del 2c.
base = DecisionTreeClassifier(random_state=SEMILLA)
base.fit(X_train, y_train)

print("profundidad:", base.get_depth(), "  hojas:", base.get_n_leaves())
# Muestro train y test lado a lado: la diferencia entre los dos es parte del diagnóstico
print(pd.DataFrame({"train": evaluar(base, X_train, y_train),
                    "test": evaluar(base, X_test, y_test)}).round(3))
matriz(base, X_test, y_test)

profundidad: 23   hojas: 128
               train   test
accuracy         1.0  0.786
balanced_acc     1.0  0.778
precision_GBM    1.0  0.750
recall_GBM       1.0  0.729
f1_GBM           1.0  0.739


,pred LGG,pred GBM
real LGG,81,17
real GBM,19,51


In [5]:
# El árbol completo no se puede leer, así que muestro solo los 3 primeros niveles.
# "truncated branch" quiere decir que sigue más abajo.
print(export_text(base, feature_names=list(X.columns), max_depth=3,
                  class_names=["LGG", "GBM"]))

|--- IDH1 <= 0.50
|   |--- IDH2 <= 0.50
|   |   |--- edad_diagnostico <= 67.31
|   |   |   |--- NF1 <= 0.50
|   |   |   |   |--- truncated branch of depth 20
|   |   |   |--- NF1 >  0.50
|   |   |   |   |--- truncated branch of depth 9
|   |   |--- edad_diagnostico >  67.31
|   |   |   |--- EGFR <= 0.50
|   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- EGFR >  0.50
|   |   |   |   |--- truncated branch of depth 5
|   |--- IDH2 >  0.50
|   |   |--- GRIN2A <= 0.50
|   |   |   |--- class: LGG
|   |   |--- GRIN2A >  0.50
|   |   |   |--- class: GBM
|--- IDH1 >  0.50
|   |--- TP53 <= 0.50
|   |   |--- PIK3R1 <= 0.50
|   |   |   |--- class: LGG
|   |   |--- PIK3R1 >  0.50
|   |   |   |--- ATRX <= 0.50
|   |   |   |   |--- class: LGG
|   |   |   |--- ATRX >  0.50
|   |   |   |   |--- class: GBM
|   |--- TP53 >  0.50
|   |   |--- raza_Blanca <= 0.50
|   |   |   |--- edad_diagnostico <= 43.74
|   |   |   |   |--- truncated branch of depth 5
|   |   |   |--- edad_diagnostico >  4

## c. Exploración de configuraciones (CV de 5 folds sobre train)

In [6]:
# Folds fijos: todas las configuraciones se evalúan sobre las mismas 5 particiones
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
METRICAS = {"accuracy": "accuracy", "balanced_acc": "balanced_accuracy",
            "precision_GBM": "precision", "recall_GBM": "recall", "f1_GBM": "f1"}

# Acá se guarda cada corrida en el momento en que se hace.
# Es un dict por nombre: si volvés a correr una celda, la fila se pisa y no se duplica.
corridas = {}

def probar(nombre, **params):
    """Evalúa una configuración con CV sobre train, la agrega a la tabla y la muestra.
    El test no se toca acá: se usa una sola vez, al final, con la elegida."""
    modelo = DecisionTreeClassifier(random_state=SEMILLA, **params)
    cv = cross_validate(modelo, X_train, y_train, cv=folds, scoring=METRICAS)
    modelo.fit(X_train, y_train)  # solo para saber el tamaño del árbol
    fila = {"params": params or "por defecto"}
    for m in METRICAS:
        fila[m] = cv[f"test_{m}"].mean()
    # El desvío entre folds dice cuánto se mueve la métrica solo por cambiar los datos
    fila["f1_GBM_desvio"] = cv["test_f1_GBM"].std()
    fila["profundidad"] = modelo.get_depth()
    fila["hojas"] = modelo.get_n_leaves()
    corridas[nombre] = fila
    return pd.DataFrame.from_dict(corridas, orient="index").round(3)

probar("base")

,params,accuracy,balanced_acc,precision_GBM,recall_GBM,f1_GBM,f1_GBM_desvio,profundidad,hojas
base,por defecto,0.797,0.792,0.757,0.762,0.759,0.025,23,128


In [7]:
probar("depth2", max_depth=2)
probar("depth3", max_depth=3)
probar("depth4", max_depth=4)
probar("depth6", max_depth=6)
probar("depth10", max_depth=10)

,params,accuracy,balanced_acc,precision_GBM,recall_GBM,f1_GBM,f1_GBM_desvio,profundidad,hojas
base,por defecto,0.797,0.792,0.757,0.762,0.759,0.025,23,128
depth2,{'max_depth': 2},0.869,0.877,0.797,0.926,0.856,0.015,2,4
depth3,{'max_depth': 3},0.866,0.875,0.790,0.929,0.853,0.017,3,8
depth4,{'max_depth': 4},0.863,0.866,0.810,0.883,0.844,0.020,4,13
depth6,{'max_depth': 6},0.842,0.846,0.779,0.872,0.822,0.030,6,30
depth10,{'max_depth': 10},0.815,0.814,0.765,0.808,0.785,0.034,10,70


In [8]:
probar("leaf2", min_samples_leaf=2)
probar("leaf5", min_samples_leaf=5)
probar("leaf10", min_samples_leaf=10)
probar("leaf20", min_samples_leaf=20)

,params,accuracy,balanced_acc,precision_GBM,recall_GBM,f1_GBM,f1_GBM_desvio,profundidad,hojas
base,por defecto,0.797,0.792,0.757,0.762,0.759,0.025,23,128
depth2,{'max_depth': 2},0.869,0.877,0.797,0.926,0.856,0.015,2,4
depth3,{'max_depth': 3},0.866,0.875,0.790,0.929,0.853,0.017,3,8
depth4,{'max_depth': 4},0.863,0.866,0.810,0.883,0.844,0.020,4,13
depth6,{'max_depth': 6},0.842,0.846,0.779,0.872,0.822,0.030,6,30
depth10,{'max_depth': 10},0.815,0.814,0.765,0.808,0.785,0.034,10,70
leaf2,{'min_samples_leaf': 2},0.821,0.808,0.828,0.727,0.773,0.022,20,101
leaf5,{'min_samples_leaf': 5},0.832,0.827,0.802,0.797,0.797,0.041,15,64
leaf10,{'min_samples_leaf': 10},0.848,0.846,0.811,0.833,0.821,0.027,13,37
leaf20,{'min_samples_leaf': 20},0.838,0.837,0.796,0.834,0.811,0.024,8,24


In [9]:
probar("d2_entropy", max_depth=2, criterion="entropy")
probar("d3_entropy", max_depth=3, criterion="entropy")
probar("d4_leaf10", max_depth=4, min_samples_leaf=10)

,params,accuracy,balanced_acc,precision_GBM,recall_GBM,f1_GBM,f1_GBM_desvio,profundidad,hojas
base,por defecto,0.797,0.792,0.757,0.762,0.759,0.025,23,128
depth2,{'max_depth': 2},0.869,0.877,0.797,0.926,0.856,0.015,2,4
depth3,{'max_depth': 3},0.866,0.875,0.790,0.929,0.853,0.017,3,8
depth4,{'max_depth': 4},0.863,0.866,0.810,0.883,0.844,0.020,4,13
depth6,{'max_depth': 6},0.842,0.846,0.779,0.872,0.822,0.030,6,30
depth10,{'max_depth': 10},0.815,0.814,0.765,0.808,0.785,0.034,10,70
leaf2,{'min_samples_leaf': 2},0.821,0.808,0.828,0.727,0.773,0.022,20,101
leaf5,{'min_samples_leaf': 5},0.832,0.827,0.802,0.797,0.797,0.041,15,64
leaf10,{'min_samples_leaf': 10},0.848,0.846,0.811,0.833,0.821,0.027,13,37
leaf20,{'min_samples_leaf': 20},0.838,0.837,0.796,0.834,0.811,0.024,8,24


In [10]:
# Configuración elegida con la regla del punto 4: mayor F1 de GBM en CV y, si hay
# empate práctico (diferencia menor al desvío entre folds), el árbol más chico.
PARAMS_ELEGIDOS = {"max_depth": 2}

elegido = DecisionTreeClassifier(random_state=SEMILLA, **PARAMS_ELEGIDOS)
elegido.fit(X_train, y_train)
print("profundidad:", elegido.get_depth(), "  hojas:", elegido.get_n_leaves())

# Comparación contra el árbol base, las dos sobre el mismo test
print(pd.DataFrame({"base": evaluar(base, X_test, y_test),
                    "elegido": evaluar(elegido, X_test, y_test)}).round(3))
# Train contra test del elegido: si se parecen, no está sobreajustando
print(pd.DataFrame({"train": evaluar(elegido, X_train, y_train),
                    "test": evaluar(elegido, X_test, y_test)}).round(3))
print(matriz(elegido, X_test, y_test))
# show_weights muestra cuántos LGG y GBM de train caen en cada hoja
print(export_text(elegido, feature_names=list(X.columns),
                  class_names=["LGG", "GBM"], show_weights=True))

profundidad: 2   hojas: 4
                base  elegido
accuracy       0.786    0.869
balanced_acc   0.778    0.880
precision_GBM  0.750    0.786
recall_GBM     0.729    0.943
f1_GBM         0.739    0.857
               train   test
accuracy       0.869  0.869
balanced_acc   0.877  0.880
precision_GBM  0.796  0.786
recall_GBM     0.926  0.943
f1_GBM         0.856  0.857
          pred LGG  pred GBM
real LGG        80        18
real GBM         4        66
|--- IDH1 <= 0.50
|   |--- IDH2 <= 0.50
|   |   |--- weights: [67.00, 261.00] class: GBM
|   |--- IDH2 >  0.50
|   |   |--- weights: [19.00, 2.00] class: LGG
|--- IDH1 >  0.50
|   |--- TP53 <= 0.50
|   |   |--- weights: [139.00, 1.00] class: LGG
|   |--- TP53 >  0.50
|   |   |--- weights: [164.00, 18.00] class: LGG



## Conclusiones.

### Qué tomé como "mejor"

Elegí el F1 de la clase GBM como métrica principal. GBM es el tumor agresivo y el error
caro es clasificar un GBM como LGG, pero tampoco quiero un modelo que diga GBM a todos.
El F1 balancea las dos cosas. Si dos configuraciones daban parecido, elegí el árbol más
chico, que es más fácil de interpretar.

Para comparar las configuraciones usé validación cruzada de 5 folds sobre el train. El
test lo usé una sola vez, al final, con la configuración elegida. Con 168 filas de test,
cada caso mueve la métrica casi un punto, y elegir mirando el test hubiera sido elegir
en parte por azar.

### Árbol base (2b)

Con los parámetros por defecto el árbol llegó a profundidad 23 y 128 hojas. En train da
1,0 en todas las métricas y en test baja a 0,786 de accuracy y 0,739 de F1. El árbol
memorizó el train: está sobreajustado.

### Lo que probé

Probé 13 configuraciones. Las tabulé en la tabla de arriba.

- Limitar la profundidad fue lo que más mejoró. El F1 en CV sube a medida que el árbol
  es más chico: 0,785 con profundidad 10 y 0,856 con profundidad 2.
- Subir `min_samples_leaf` también mejora contra la base, pero menos. El mejor fue 10
  hojas mínimas, con 0,821.
- Cambiar gini por entropy no cambió nada en profundidad 2 y casi nada en 3.
- Las mejores fueron profundidad 2 (0,856) y profundidad 3 (0,853). La diferencia es
  0,003, más chica que el desvío entre folds (0,015), así que las tomo como empatadas.
  Elegí profundidad 2 por ser el árbol más chico: 4 hojas.

### Resultado en test

| | base | elegido (max_depth=2) |
| --- | --- | --- |
| accuracy | 0,786 | 0,869 |
| recall GBM | 0,729 | 0,943 |
| F1 GBM | 0,739 | 0,857 |

Los GBM clasificados como LGG bajaron de 19 a 4. A cambio, los LGG clasificados como
GBM pasaron de 17 a 18. Para este problema es un buen intercambio. Además, el F1 en train
(0,856) y en test (0,857) son casi iguales, así que este árbol no está sobreajustando.

### Qué dice el árbol

Con dos genes alcanza: si IDH1 e IDH2 no están mutados, el árbol dice GBM. En cualquier
otro caso dice LGG. El corte por TP53 deja LGG en las dos ramas, así que no cambia la
predicción. Eso tiene que ver con el objetivo de la ficha de UCI: un árbol de 4 hojas que
usa 2 genes clasifica mejor que uno de 128 hojas que usa todo.

Una limitación es que el test tiene solo 70 GBM, así que el recall de 0,943 es 66 de 70.
Con otro split podría moverse unos puntos.